In [ ]:
# Placeholder parameters: illustrative values only, not the ones used in live research.
WINDOW = 20  # 個人智慧財產,參數不提供 (placeholder)
ENTRY_Z_LONG = -2.0  # 個人智慧財產,參數不提供 (placeholder)
ENTRY_Z_SHORT = 2.0  # 個人智慧財產,參數不提供 (placeholder)
EXIT_Z_LONG = -0.5  # 個人智慧財產,參數不提供 (placeholder)
EXIT_Z_SHORT = 0.5  # 個人智慧財產,參數不提供 (placeholder)
STACK_INTERVAL = 1.0  # 個人智慧財產,參數不提供 (placeholder)
MAX_POSITIONS = 3  # 個人智慧財產,參數不提供 (placeholder)
MAX_HOLD_DAYS_ = 20  # 個人智慧財產,參數不提供 (placeholder)
NOTIONAL = 1  # 個人智慧財產,參數不提供 (placeholder)

# Pairs Trading Strategy Notebook

This notebook implements a basic pairs trading strategy based on the concept of mean reversion. It analyzes the spread between two related assets (Alphabet A and Alphabet C stock prices in this example) and generates trading signals based on Z-scores of this spread.

In [ ]:
# @title
# Step X: Import libraries
import io
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

## X Data Loading

This section loads historical stock price data for two assets (Alphabet A and Alphabet C) from CSV files into pandas DataFrames. It then prints the columns and displays the first few rows of each DataFrame for initial inspection.

In [ ]:
a_file = "Alphabet A Stock Price History.csv"   # Asset A
c_file = "Alphabet C Stock Price History.csv"    # Asset C

df_a = pd.read_csv(a_file)
df_c = pd.read_csv(c_file)

print("A columns:", df_a.columns.tolist())
print("C columns:", df_c.columns.tolist())

df_a.head(), df_c.head()

## X Data Preparation

This section cleans and prepares the loaded data:
- It standardizes column names.
- Selects only the 'Date' and 'Price' columns.
- Removes commas from price strings and converts them to float.
- Converts the 'Date' column to datetime objects.
- Sorts the DataFrames by date.
- Merges the two DataFrames on 'Date' to create a combined DataFrame.
- Calculates the 'Spread' as the natural logarithm of the ratio of the two asset prices (Log Ratio).

**Note**: The original code expects `Alphabet A Stock Price History.csv` and `Alphabet C Stock Price History.csv` to be present. If you receive a `FileNotFoundError`, ensure these files are uploaded to your Colab environment.

In [ ]:
df_a.columns = [c.strip() for c in df_a.columns]
df_c.columns = [c.strip() for c in df_c.columns]

date_col  = "Date"
price_col = "Price"

df_a = df_a[[date_col, price_col]].copy()
df_c = df_c[[date_col, price_col]].copy()

df_a.columns = ["Date", "A_Close"]
df_c.columns = ["Date", "C_Close"]

# ── Strip commas and convert to float ──
df_a["A_Close"] = df_a["A_Close"].astype(str).str.replace(",", "").astype(float)
df_c["C_Close"] = df_c["C_Close"].astype(str).str.replace(",", "").astype(float)

df_a["Date"] = pd.to_datetime(df_a["Date"])
df_c["Date"] = pd.to_datetime(df_c["Date"])

df_a = df_a.sort_values("Date").reset_index(drop=True)
df_c = df_c.sort_values("Date").reset_index(drop=True)

df = pd.merge(df_a, df_c, on="Date", how="inner").sort_values("Date").reset_index(drop=True)

# Normalized: log ratio
df["Spread"] = np.log(df["A_Close"] / df["C_Close"])

df.head()

## X Spread Calculation and Z-Score

This section calculates rolling statistics (mean and standard deviation) of the spread over a defined `window` (e.g., X days). It then computes the Z-score for the spread, which measures how many standard deviations the current spread is from its rolling mean.

Two plots are generated:
X  **Log Ratio and Rolling Mean**: Visualizes the spread and its X-day rolling mean over time.
X  **X-Day Z-score**: Shows the Z-score of the spread, highlighting key thresholds (-X, -X, -X and X, X, X) often used in mean-reversion strategies.

In [ ]:
window = WINDOW

df["Spread_Mean_60"] = df["Spread"].rolling(window).mean()
df["Spread_Std_60"]  = df["Spread"].rolling(window).std()
df["ZScore_60"]      = (df["Spread"] - df["Spread_Mean_60"]) / df["Spread_Std_60"]

# Plot X: Log ratio + rolling mean
plt.figure(figsize=(12, 5))
plt.plot(df["Date"], df["Spread"],         label="Log Ratio (A/C)")
plt.plot(df["Date"], df["Spread_Mean_60"], label="60D Rolling Mean", linestyle="--")
# plt.axhline(X, linestyle=":")
plt.title("Log Ratio and 60-Day Rolling Mean")
plt.xlabel("Date")
plt.ylabel("Log Ratio (A/C)")
plt.legend()
plt.show()

# Plot X: Z-score
plt.figure(figsize=(12, 5))
plt.plot(df["Date"], df["ZScore_60"], label="60D Z-score")
plt.axhline(0,  linestyle=":")
plt.axhline(2,  linestyle="--", color="red",     alpha=0.6)
plt.axhline(3,  linestyle="--", color="orange",  alpha=0.6)
plt.axhline(4,  linestyle="--", color="darkred", alpha=0.6)
plt.axhline(-2, linestyle="--", color="red",     alpha=0.6)
plt.axhline(-3, linestyle="--", color="orange",  alpha=0.6)
plt.title("Log Ratio 60-Day Z-score Over Time")
plt.xlabel("Date")
plt.ylabel("Z-score")
plt.legend()
plt.show()

## X Summary Statistics and Latest Z-Score

This section provides summary statistics for the entire sample's log ratio (mean and standard deviation) and displays the latest values for the date, log ratio, X-day rolling mean, X-day standard deviation, and X-day Z-score.

In [ ]:
# @title
print("Full-sample mean log ratio:", df["Spread"].mean())
print("Full-sample std  log ratio:", df["Spread"].std())

latest = df.iloc[-1]
print("\nLatest date:       ", latest["Date"].date())
print("Latest log ratio:  ", latest["Spread"])
print("Latest 60d mean:   ", latest["Spread_Mean_60"])
print("Latest 60d std:    ", latest["Spread_Std_60"])
print("Latest 60d z-score:", latest["ZScore_60"])

## X Mean Reversion (AR(X)) Analysis

This section estimates the parameters of an Autoregressive model of order X (AR(X)) for the spread. This helps in understanding the mean-reverting nature of the spread. It calculates the `phi` (b) coefficient, which indicates the speed of mean reversion, and estimates the half-life of the spread – the time it takes for the spread to revert halfway to its mean.

In [ ]:
# @title
spread = df["Spread"].dropna().copy()
lagged = spread.shift(1)

ar_df = pd.DataFrame({
    "Spread_t":    spread,
    "Spread_lag1": lagged
}).dropna()

X = ar_df["Spread_lag1"].values
Y = ar_df["Spread_t"].values

b, a = np.polyfit(X, Y, 1)

print("AR(1) intercept a:", a)
print("AR(1) phi b:",       b)

if 0 < b < 1:
    half_life = np.log(0.5) / np.log(b)
    print("Estimated half-life (trading days):", half_life)
else:
    print("Half-life not defined — phi is not between 0 and 1.")

## X Strategy Parameters

This section defines the key parameters for the pairs trading strategy:
- `entry_z_long`, `entry_z_short`: Z-score thresholds for entering long and short spread positions.
- `exit_z_long`, `exit_z_short`: Z-score thresholds for exiting long and short spread positions.
- `interval`: Minimum Z-score difference required for stacking new positions.
- `max_positions`: Maximum number of stacked positions allowed.
- `MAX_HOLD_DAYS`: Maximum number of days to hold an open position before a time stop exit.
- `direction_filter`: Option to filter trades to 'long', 'short', or 'both'.
- `notional`: Notional value per trade leg.

In [ ]:
entry_z_long = ENTRY_Z_LONG
entry_z_short = ENTRY_Z_SHORT

exit_z_long = EXIT_Z_LONG
exit_z_short = EXIT_Z_SHORT

interval = STACK_INTERVAL
max_positions = MAX_POSITIONS
MAX_HOLD_DAYS = MAX_HOLD_DAYS_

direction_filter = "both"   # options: "long", "short", "both"
notional = NOTIONAL

## X Backtesting Logic

This is the core backtesting loop. It iterates through the historical data, calculates rolling mean and standard deviation for the spread, and determines the Z-score at each point. It then applies the defined entry and exit rules for the pairs trading strategy, including position stacking and time stops. It records open positions and completed trades.

In [ ]:
# @title
bt = df.copy().reset_index(drop=True)
bt["PnL"]      = 0.0
bt["Position"] = 0
bt["Z"]        = np.nan

positions = []   # list of open positions (each = dict)
trades    = []

for i in range(window, len(bt)):
    spread_hist = bt.loc[i-window:i-1, "Spread"]

    mu    = spread_hist.mean()
    sigma = spread_hist.std()

    if pd.isna(sigma) or sigma == 0:
        continue

    z = (bt.loc[i, "Spread"] - mu) / sigma
    bt.loc[i, "Z"] = z

    # -------------------------
    # ENTRY (STACKING LOGIC)
    # -------------------------
    if len(positions) < max_positions:

        # SHORT stacking
        if z > entry_z_short:
            if len(positions) == 0 or (
                positions[-1]["Direction"] == "Short" and
                z >= positions[-1]["Entry_Z"] + interval
            ):
                positions.append({
                    "Direction": "Short",
                    "Entry_Idx": i,
                    "Entry_Z":   z
                })

        # LONG stacking
        elif z < entry_z_long:
            if len(positions) == 0 or (
                positions[-1]["Direction"] == "Long" and
                z <= positions[-1]["Entry_Z"] - interval
            ):
                positions.append({
                    "Direction": "Long",
                    "Entry_Idx": i,
                    "Entry_Z":   z
                })

    # -------------------------
    # POSITION (aggregate)
    # -------------------------
    net_position = sum([-1 if p["Direction"]=="Short" else 1 for p in positions])
    bt.loc[i, "Position"] = net_position

    # -------------------------
    # PnL (aggregate exposure)
    # -------------------------
    if i > 0:
        dS = bt.loc[i, "Spread"] - bt.loc[i-1, "Spread"]
        bt.loc[i, "PnL"] = net_position * dS

    # -------------------------
    # EXIT CONDITIONS
    # -------------------------
    exit_flag   = False
    exit_reason = None

    if len(positions) > 0:
        first_entry_idx = positions[0]["Entry_Idx"]

        # Signal exit (close ALL)
        if net_position < 0 and z <= exit_z_short:
            exit_flag   = True
            exit_reason = "Signal Exit"
        elif net_position > 0 and z >= exit_z_long:
            exit_flag   = True
            exit_reason = "Signal Exit"

        # Time stop (based on oldest position)
        elif (i - first_entry_idx) >= MAX_HOLD_DAYS:
            exit_flag   = True
            exit_reason = "Time Stop"

    # -------------------------
    # EXECUTE EXIT
    # -------------------------
    if exit_flag and len(positions) > 0:

        # record ONE aggregated trade
        trades.append({
            "Direction":    "Long Spread" if net_position > 0 else "Short Spread",
            "Entry_Date":   bt.loc[positions[0]["Entry_Idx"], "Date"],
            "Exit_Date":    bt.loc[i, "Date"],
            "Entry_Z":      positions[0]["Entry_Z"],
            "Exit_Z":       z,
            "Entry_Spread": bt.loc[positions[0]["Entry_Idx"], "Spread"],
            "Exit_Spread":  bt.loc[i, "Spread"],
            "Holding_Days": i - positions[0]["Entry_Idx"],
            "Exit_Reason":  exit_reason,
            "Num_Legs":     len(positions),

            # 🔥 THIS IS THE FIX
            "Entries": [
                {
                    "Entry_Date": bt.loc[p["Entry_Idx"], "Date"],
                    "Entry_Z":    p["Entry_Z"],
                    "Entry_Idx":  p["Entry_Idx"]
                }
                for p in positions
            ]
        })

        positions = []

## X PnL Calculation and Trade Analysis

This section processes the recorded trades. It defines a function `compute_pnl` to calculate the profit and loss (PnL) for each trade, taking into account the stacking of positions and the notional value. It then calculates various performance metrics, including:
- Total number of trades.
- Win rate.
- Total PnL in dollars.
- Average PnL per trade.
- Average return percentage per trade.

The results are displayed in a DataFrame, which can be filtered by `direction_filter` (long, short, or both).

In [ ]:
# @title
trades_df = pd.DataFrame(trades)

print("=== Direction Definitions ===")
print("Long Spread  = Long A  / Short C")
print("Short Spread = Short A / Long C")
print()

# ── Filter input ──


def compute_pnl(row):
    entry_date = row["Entry_Date"]
    exit_date  = row["Exit_Date"]

    exit_a = bt.loc[bt["Date"] == exit_date, "A_Close"].values[0]
    exit_c = bt.loc[bt["Date"] == exit_date, "C_Close"].values[0]

    entry_rows = bt[
        (bt["Date"] >= entry_date) &
        (bt["Date"] <= exit_date)
    ].copy()

    entry_rows = entry_rows.iloc[:row["Num_Legs"]]

    pnl_total = 0

    for _, er in entry_rows.iterrows():
        entry_a = er["A_Close"]
        entry_c = er["C_Close"]

        shares_a = notional / entry_a
        shares_c = notional / entry_c

        if row["Direction"] == "Long Spread":
            pnl = shares_a * (exit_a - entry_a) - shares_c * (exit_c - entry_c)
        else:
            pnl = -shares_a * (exit_a - entry_a) + shares_c * (exit_c - entry_c)

        pnl_total += pnl

    return pnl_total


trades_df["PnL_Dollar"] = trades_df.apply(compute_pnl, axis=1)
trades_df["Return_pct"] = trades_df["PnL_Dollar"] / (2 * notional * trades_df["Num_Legs"]) * 100

# ── Apply filter ──
if direction_filter == "long":
    view_df = trades_df[trades_df["Direction"] == "Long Spread"]
elif direction_filter == "short":
    view_df = trades_df[trades_df["Direction"] == "Short Spread"]
else:
    view_df = trades_df

print(f"Filter:                {direction_filter.upper()}")
print(f"Notional per leg:      ${notional:,}")
print(f"Total trades:          {len(view_df)}")
print(f"Win rate:              {(view_df['PnL_Dollar'] > 0).mean():.0%}")
print(f"Total PnL:             ${view_df['PnL_Dollar'].sum():.2f}")
print(f"Avg PnL per trade:     ${view_df['PnL_Dollar'].mean():.2f}")
print(f"Avg Return per trade:  {view_df['Return_pct'].mean():.4f}%")

display(view_df)

## X Performance Metrics

This section calculates additional performance metrics:
- Total PnL for all trades.
- Total holding days across all trades.
- Daily return based on deployed capital.
- Annualized return (assuming X trading days in a year).

In [ ]:
# @title
total_pnl = trades_df["PnL_Dollar"].sum()
total_days = trades_df["Holding_Days"].sum()

# Return per day (on deployed capital)
daily_return = total_pnl / (2 * notional) / total_days

# Annualized (X trading days)
annualized_return = (1 + daily_return) ** 252 - 1

print(f"Total PnL: ${total_pnl:.4f}")
print(f"Total Holding Days: {total_days}")
print(f"Daily Return: {daily_return*100:.4f}%")
print(f"Annualized Return: {annualized_return*100:.2f}%")

## X Visualization of Trades

This section generates two plots to visualize the trading activity against the spread and Z-score:
X  **Log Ratio and Z-score (Dual Axis)**: Displays the log ratio and its Z-score over time, with horizontal lines indicating the entry and exit thresholds for both long and short positions.
X  **Log Ratio with Stacked Trade Markers**: Plots the log ratio with markers indicating long entries (green triangles), short entries (red inverted triangles), and exits (black circles). This visualization helps in understanding where trades were initiated and closed relative to the spread's movement and Z-score.

In [ ]:
# @title

# =========================
# Plot X: Log ratio + Z-score (UNCHANGED)
# =========================
fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.plot(bt["Date"], bt["Spread"], label="Log Ratio (A/C)")
ax1.set_ylabel("Log Ratio (A/C)")

ax2 = ax1.twinx()
ax2.plot(bt["Date"], bt["Z"], linestyle="--", color="red", label="Z-score")
ax2.set_ylabel("Z-score")

ax2.axhline(entry_z_short, linestyle="--", color="red",  alpha=0.5)
ax2.axhline(entry_z_long,  linestyle="--", color="blue", alpha=0.5)
ax2.axhline(exit_z_short,  linestyle=":",  color="red",  alpha=0.5)
ax2.axhline(exit_z_long,   linestyle=":",  color="blue", alpha=0.5)

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2)

plt.title("Log Ratio and Z-score (Dual Axis)")
plt.show()


# =========================
# Plot X: Log ratio with STACKED trade markers
# =========================
plt.figure(figsize=(12, 5))
plt.plot(bt["Date"], bt["Spread"], label="Log Ratio (A/C)")

if len(trades_df) > 0:
    spread_series = bt.set_index("Date")["Spread"]

    long_x, long_y = [], []
    short_x, short_y = [], []
    exit_x, exit_y = [], []

    for _, row in trades_df.iterrows():

        # -------- MULTIPLE ENTRIES --------
        if "Entries" in row and isinstance(row["Entries"], list):

            for j, entry in enumerate(row["Entries"]):
                entry_date = entry["Entry_Date"]

                if entry_date in spread_series.index:
                    y_val = spread_series.loc[entry_date]

                    # Optional: scale marker size by stacking depth
                    size_scale = 60 + j * 30

                    if row["Direction"] == "Long Spread":
                        plt.scatter(entry_date, y_val,
                                    marker="^", s=size_scale,
                                    color="green", alpha=0.7)
                    else:
                        plt.scatter(entry_date, y_val,
                                    marker="v", s=size_scale,
                                    color="red", alpha=0.7)

        # -------- EXIT (ONE PER TRADE) --------
        exit_date = row["Exit_Date"]
        if exit_date in spread_series.index:
            exit_x.append(exit_date)
            exit_y.append(spread_series.loc[exit_date])

    # Plot exits
    plt.scatter(exit_x, exit_y,
                marker="o", s=70,
                label="Exit", color="black")

# Legend (manual for clarity)
plt.scatter([], [], marker="^", color="green", label="Long Entries")
plt.scatter([], [], marker="v", color="red",   label="Short Entries")
plt.scatter([], [], marker="o", color="black", label="Exit")

plt.title("Log Ratio with Stacked Trade Markers")
plt.xlabel("Date")
plt.ylabel("Log Ratio (A/C)")
plt.legend()
plt.show()

## X Live Z-Score Calculation with Yfinance

This section demonstrates how to calculate the current Z-score for a given pair of assets using live data fetched from `yfinance`. It downloads recent price data, computes the spread, and then calculates the Z-score based on the rolling statistics from the historical data (the `mu` and `sigma` values computed in the backtesting section). It then provides a trading signal based on the `entry_z_long` and `entry_z_short` thresholds.

In [ ]:
# =========================
# LIVE Z-SCORE + PRICE DISPLAY
# =========================
import yfinance as yf
import numpy as np
import pandas as pd

ticker_a = "GOOG"
ticker_b = "GOOGL"

window = WINDOW

# ---- download ----
data = yf.download(
    [ticker_a, ticker_b],
    period="6mo",
    interval="1d",
    auto_adjust=True
)
# print(data)
close = data["Close"].dropna()
close.columns = [f"{ticker_b}_Close", f"{ticker_a}_Close"]

# ---- compute spread ----
close["Spread"] = np.log(close[f"{ticker_a}_Close"] / close[f"{ticker_b}_Close"])

# ---- rolling stats ----
spread_hist = close["Spread"].iloc[-window:]
mu = spread_hist.mean()
sigma = spread_hist.std()

# ---- current values ----
current = close.iloc[-1]
current_date = close.index[-1]

price_a = 100.0
price_b = 100.0
# print(price_a)

current_spread = current["Spread"]
current_z = (current_spread - mu) / sigma

print(f"Date:        {current_date}")
print(f"{ticker_a} Price: {price_a:,.2f}")
print(f"{ticker_b} Price: {price_b:,.2f}")
print(f"Spread:      {current_spread:.6f}")
print(f"Z-score:     {current_z:.4f}")
print(f"Mean (60d):  {mu:.6f}")
print(f"Std (60d):   {sigma:.6f}")

# ---- signal ----
if current_z < entry_z_long:
    print(">>> SIGNAL: LONG SPREAD ("+ ticker_a +" cheap)")
elif current_z > entry_z_short:
    print(">>> SIGNAL: SHORT SPREAD (if possible)")
elif exit_z_long <= current_z <= exit_z_short:
    print(">>> SIGNAL: EXIT / NEUTRAL")
else:
    print(">>> NO ACTION")

## X Manual Input Z-Score Calculator

This section allows for manual input of asset prices to calculate the spread and Z-score. It uses the `mu` and `sigma` values derived from the historical data's X-day rolling statistics, providing a way to test signals with hypothetical current prices. It also outputs a trading signal based on the defined strategy parameters.

In [ ]:
# =========================
# MANUAL INPUT Z-SCORE CALCULATOR
# =========================

# ---- user input ----
price_a = 100.0
price_b = 100.0

# ---- compute spread ----
manual_spread = np.log(price_a / price_b)

# ---- use SAME rolling stats from your model ----
# (must already exist from previous block)
# mu, sigma from X-day window

manual_z = (manual_spread - mu) / sigma**0.5
# Note: `mu` and `sigma` are from the previous `yfinance` call with BND and AGG.
# For a true manual calculation based on GOOG/GOOGL, you would need to run the previous cell first.
# print(mu, sigma) # Uncomment to see the BND/AGG mu and sigma

print("\n===== MANUAL INPUT RESULT =====")
print(f"{ticker_a} Price: {price_a:,.2f}")
print(f"{ticker_b} Price: {price_b:,.2f}")
print(f"Spread:      {manual_spread:.6f}")
print(f"Z-score:     {manual_z:.4f}")

# ---- signal ----
if manual_z < entry_z_long:
    print(">>> SIGNAL: LONG SPREAD ("+ticker_a+" cheap)")
elif manual_z > entry_z_short:
    print(">>> SIGNAL: SHORT SPREAD ("+ticker_b+" cheap)")
elif exit_z_long <= manual_z <= exit_z_short:
    print(">>> SIGNAL: EXIT / NEUTRAL")
else:
    print(">>> NO ACTION")